# Coding agent: what can `coding-agent-sa` actually do in BigQuery?

Every call below runs **as the coding agent's service account**, by impersonating it, not as you. It answers the open question in `docs/coding-agent.md`: does a **dry run** work with metadata access only, or does it need read access to the tables it references?

Nothing here executes a query. The dry runs are validated by BigQuery and never run, and no rows are read. Needs: your gcloud login (the same one the other notebooks use), and the setup in `local-dev-environment-setup.md`, Step 19.

Run the two code cells top to bottom, then paste the printed output.

In [1]:
import sys
sys.path.insert(0, "..")

import google.auth
from google.api_core.exceptions import Forbidden
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request
from google.cloud import bigquery

from app.config import AGENT_SAFE_DATASET, CHAT_HISTORY_DATASET, CODING_AGENT_DATASETS, CODING_AGENT_SA_EMAIL, GCP_PROJECT_ID, TELEMETRY_DATASET

# Every call below runs as the coding agent's service account, not as you
source_credentials, _ = google.auth.default()
credentials = impersonated_credentials.Credentials(
    source_credentials=source_credentials, target_principal=CODING_AGENT_SA_EMAIL,
    target_scopes=["https://www.googleapis.com/auth/bigquery"], lifetime=3600)
credentials.refresh(Request())  # fails here, with the reason, if impersonation is not set up
client = bigquery.Client(project=GCP_PROJECT_ID, credentials=credentials)
print("running as:", credentials.service_account_email)

running as: coding-agent-sa@instacart-ml-model.iam.gserviceaccount.com


In [5]:
def check(label, action):
    """Run one BigQuery call as the service account and print what happened. Returns (result, error message)."""
    try:
        result = action()
    except Forbidden as error:
        print(f"  DENIED   {label}: {error.message[:240]}")
        return None, error.message
    except Exception as error:
        print(f"  ERROR    {label}: {type(error).__name__}: {str(error)[:240]}")
        return None, str(error)
    print(f"  OK       {label}")
    return result, None


print("1. Tables in the allowed datasets (expect OK)")
first_tables = {}
for dataset in CODING_AGENT_DATASETS:
    tables, _ = check(f"list tables in {dataset}", lambda: list(client.list_tables(f"{GCP_PROJECT_ID}.{dataset}")))
    if tables:
        first_tables[dataset] = tables[0]
        print(f"           {len(tables)} tables, for example: {', '.join(table.table_id for table in tables[:4])}")

print("\n2. Schemas and column descriptions (expect OK)")
for dataset, listed in first_tables.items():
    table, _ = check(f"read the schema of {dataset}.{listed.table_id}", lambda: client.get_table(listed.reference))
    if table:
        described = sum(1 for field in table.schema if field.description)
        print(f"           {len(table.schema)} columns, {described} with a description; table description present: {bool(table.description)}")

print("\n3. Datasets the service account must NOT reach (expect DENIED)")
for dataset in (AGENT_SAFE_DATASET, TELEMETRY_DATASET, CHAT_HISTORY_DATASET):
    reached, _ = check(f"list tables in {dataset}", lambda: list(client.list_tables(f"{GCP_PROJECT_ID}.{dataset}")))
    if reached is not None:
        print("           UNEXPECTED: this access should not exist")

print("\n4. Permissions the service account holds on one table")
held = []
sample = next(iter(first_tables.values()), None)
if sample:
    # table-level permissions only: creating a table is a dataset-level permission, which a table cannot be asked about
    wanted = ["bigquery.tables.get", "bigquery.tables.getData", "bigquery.tables.update", "bigquery.tables.updateData",
              "bigquery.tables.delete", "bigquery.tables.export"]
    response, _ = check(f"test permissions on {sample.dataset_id}.{sample.table_id}", lambda: client.test_iam_permissions(sample.reference, wanted))
    if response is not None:
        held = response.get("permissions", [])
        for permission in wanted:
            print(f"           {'held    ' if permission in held else 'not held'}  {permission}")

print("\n5. Dry runs: validated by BigQuery, never executed")
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_errors = []
for dataset, listed in first_tables.items():
    sql = f"SELECT * FROM `{listed.project}.{listed.dataset_id}.{listed.table_id}` LIMIT 1"
    job, error = check(f"dry run on {dataset}.{listed.table_id}", lambda: client.query(sql, job_config=dry_run_config))
    if job:
        print(f"           valid; would process {(job.total_bytes_processed or 0):,} bytes; {len(job.schema or [])} output columns; statement type {job.statement_type}")
    else:
        dry_run_errors.append(error)

print("\nVERDICT")
holds_read = "bigquery.tables.getData" in held
holds_write = [permission for permission in held if permission in ("bigquery.tables.update", "bigquery.tables.updateData", "bigquery.tables.delete")]
if first_tables and not dry_run_errors:
    if holds_read:
        print("  Dry runs work, and the account holds read access (bigquery.tables.getData), as granted by Data Viewer.")
    else:
        print("  A: a dry run works with metadata access only. No data-reading role is needed.")
elif any("getData" in (error or "") or "permission to query" in (error or "") for error in dry_run_errors):
    print("  B: a dry run needs permission to query (read) the referenced tables: bigquery.tables.getData.")
    print(f"     BigQuery said: {dry_run_errors[0][:300]}")
    print("     Decide: grant BigQuery Data Viewer on the three datasets, use empty replica tables, or drop the tool.")
else:
    print("  Inconclusive: a call failed for another reason. Paste the output above.")
print(f"  Write permissions held: {', '.join(holds_write) if holds_write else 'none'}")

1. Tables in the allowed datasets (expect OK)
  OK       list tables in instacart
           9 tables, for example: aisles, cleansed_orders, cleansed_products, departments
  OK       list tables in building_blocks
           8 tables, for example: base_analytical_table, candidate_skeleton, prelim_user_aisle_features, prelim_user_department_features
  OK       list tables in ml_model_feature_table
           2 tables, for example: final_ml_features_table, skeleton_features

2. Schemas and column descriptions (expect OK)
  OK       read the schema of instacart.aisles
           2 columns, 0 with a description; table description present: False
  OK       read the schema of building_blocks.base_analytical_table
           18 columns, 18 with a description; table description present: True
  OK       read the schema of ml_model_feature_table.final_ml_features_table
           39 columns, 39 with a description; table description present: True

3. Datasets the service account must NOT reach (e